# Notebook #00 — Business Understanding & Problem Formulation

**Đề tài:** *Xây dựng bộ dữ liệu chuyến bay chậm, hủy và đúng giờ của các hãng hàng không Việt Nam* (`FlightRisk VN 2.0`).

### Mục tiêu & Phạm vi
- **Bài toán chính A (Classification - 3 lớp ICAO tại `T-3h`):** Dự đoán trạng thái khởi hành/đến trước 3 giờ:
  - `0: on_time` (trễ `< 15` phút)
  - `1: delayed` (trễ `>= 15` phút)
  - `2: cancelled` (hủy chuyến)
- **Bài toán B (Regression - L1/MAE):** Dự đoán số phút trễ (`delay_min`) **chỉ trên các chuyến bay hoàn tất** (`on_time` hoặc `delayed`; tuyệt đối không gán `0` cho chuyến hủy).
- **Nghiên cứu đối chứng (Ablation):** Nhị phân (`on_time` vs `disrupted = delayed | cancelled`) và độ nhạy chân trời `T-1h / T-3h / T-6h`.
- **Thư viện chính:** `yaml`, `json`, `pathlib`, `flight_delay.common.constants`.

In [1]:
import json
from pathlib import Path
from flight_delay.common.constants import (
    AIRLINES_CFG, AIRPORTS_CFG, EXPERIMENTS_CFG, MONITORING_CFG, PROJECT_CFG, SCHEMA_CFG, SOURCES_CFG
)

out_dir = Path("artifacts/business")
out_dir.mkdir(parents=True, exist_ok=True)

project_scope = {
    "project_name": PROJECT_CFG["project"]["name"],
    "version": PROJECT_CFG["project"]["version"],
    "prediction_horizon_hours": PROJECT_CFG["prediction_horizon"]["primary_hours"],
    "ablation_horizons": PROJECT_CFG["prediction_horizon"]["ablation_hours"],
    "airlines": list(AIRLINES_CFG["vn_airlines"]),
    "major_airports": list(AIRPORTS_CFG["monitored_airports"].keys()),
    "sources": ["FIDS Airport Portals", "Aviationstack REST API", "Open-Meteo Hourly Weather"],
    "primary_classification_labels": SCHEMA_CFG["gold"]["primary_classification_labels"],
    "binary_ablation_labels": SCHEMA_CFG["gold"]["binary_ablation_labels"],
    "delay_threshold_minutes": SCHEMA_CFG["silver"]["delay_threshold_minutes"],
}
(out_dir / "project_scope.json").write_text(json.dumps(project_scope, indent=2, ensure_ascii=False), encoding="utf-8")

eval_plan = {
    "split_strategy": "Chronological 70% Train / 10% Validation / 20% Holdout Test (zero random shuffle)",
    "classification_selection_metric": EXPERIMENTS_CFG["classification"]["selection_metric"],
    "classification_metrics": ["Accuracy", "Balanced Accuracy", "Macro-F1", "Delayed Recall", "Cancelled Recall", "ROC-AUC OvR", "PR-AUC Macro", "LogLoss", "Brier Score"],
    "regression_selection_metric": EXPERIMENTS_CFG["regression"]["selection_metric"],
    "regression_metrics": ["MAE_minutes", "MedAE_minutes", "RMSE_minutes", "R2_Score"],
    "statistical_tests": ["Expanding Temporal CV", "Paired Bootstrap 95% CI", "McNemar Test", "Permutation Test (200x)"],
}
(out_dir / "evaluation_plan.json").write_text(json.dumps(eval_plan, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== PROJECT SCOPE SAVED ===")
print(json.dumps(project_scope, indent=2, ensure_ascii=False))

=== PROJECT SCOPE SAVED ===
{
  "project_name": "ScrapeData — Vietnamese Airline Flight Data Mining & Predictive Analytics Platform",
  "version": "2.0.0",
  "prediction_horizon_hours": 3,
  "ablation_horizons": [
    1,
    6
  ],
  "airlines": [
    "VN",
    "VJ",
    "QH",
    "BL",
    "VU",
    "9G",
    "0V"
  ],
  "major_airports": [
    "SGN",
    "HAN",
    "DAD",
    "CXR",
    "PQC"
  ],
  "sources": [
    "FIDS Airport Portals",
    "Aviationstack REST API",
    "Open-Meteo Hourly Weather"
  ],
  "primary_classification_labels": {
    "on_time": 0,
    "delayed": 1,
    "cancelled": 2
  },
  "binary_ablation_labels": {
    "on_time": 0,
    "disrupted": 1
  },
  "delay_threshold_minutes": 15
}
